In [ ]:
%load_ext autoreload
%autoreload 2

import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, precision_score, recall_score
import warnings
warnings.filterwarnings("ignore", category=RuntimeWarning)
from utils import load_data, DotDict
import numpy as np
from transformers import BertTokenizer, BertModel
import torch
from tqdm.auto import tqdm

In [ ]:
settings = DotDict({
    "undersample": False, # if true, undersample active/acquired classes to match failed for train set
    "include_country": True, # if true, include coountry as a feature
    "filter_tech": True, # if true, only include tech related companies w/ the following keywords
    "only_acquired": True, # if true, exclude "active" companies
})

keywords = [
    "AI",
    "Machine Learning",
    "Generative AI",
    "AI Assistant",
    "AI-Enhanced Learning",
    "AI-powered Drug Discovery",
    "AIOps",
    "Artificial Intelligence",
    "ML",
    "NLP",
    "Data Engineering",
    "Data Science",
    "Analytics",
    "Big Data",
    "Computer Vision",
    "Deep Learning",
    # "B2B",
    "Conversational AI",
    "DevOps",
    "Developer Tools"
]

dataset, X_train, X_test, y_train, y_test, y_train_orig, train_idx, test_idx, idx_to_class = load_data("updated_data", settings, keywords)
device = "cuda" if torch.cuda.is_available() else "cpu"

X_longdesc = dataset.dropna(subset=["longDescription"])
print("shape before/after dropping null longDescription", dataset.shape, X_longdesc.shape)


tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
model = BertModel.from_pretrained('bert-base-uncased')

model.to(device)

def get_bert_embeddings(texts, batch_size=32):
    embeddings = []
    for i in tqdm(range(0, len(texts), batch_size), desc="Generating BERT embeddings"):
        batch_texts = texts[i:i + batch_size]
        inputs = tokenizer(batch_texts, return_tensors='pt', padding=True, truncation=True, max_length=512).to(device)
        with torch.no_grad():
            outputs = model(**inputs)
        emb = outputs.last_hidden_state[:, 0, :].detach().cpu().numpy()
        embeddings.append(emb)
    return np.vstack(embeddings)

long_descriptions = X_longdesc["longDescription"].tolist()
bert_embeddings = get_bert_embeddings(long_descriptions)
print("BERT embeddings shape", bert_embeddings.shape)
y_longdesc = X_longdesc["status"].apply(lambda x: 0 if x == "Inactive" else 1).values
X_train_ld, X_test_ld, y_train_ld, y_test_ld = train_test_split(
    bert_embeddings, y_longdesc, test_size=0.2, random_state=42, stratify=y_longdesc
)

# TODO: train a model that predicts success based on the name...?

In [ ]:
rf_model_ld = RandomForestClassifier(n_estimators=10, max_depth=5)
rf_model_ld.fit(X_train_ld, y_train_ld)
print("--- Random Forest on BERT embeddings ---") # ideally the bert embeddings should help capture deeper semantic meaning
print("train acc", rf_model_ld.score(X_train_ld, y_train_ld))
print("test acc", rf_model_ld.score(X_test_ld, y_test_ld))
y_pred_ld = rf_model_ld.predict(X_test_ld)
print("f1 score", f1_score(y_test_ld, y_pred_ld))
print("precision", precision_score(y_test_ld, y_pred_ld))
print("recall", recall_score(y_test_ld, y_pred_ld))

In [ ]:
from utils import eval_models, RandomModel

rndm_model = RandomModel(p=sum(y_train_ld == 1)/len(y_train_ld))
print(f1_score(y_test_ld, rndm_model.predict(X_test_ld)))

lr_model = LogisticRegression(class_weight="balanced")
lr_model.fit(X_train,y_train)

eval_models({"Logistic Regression": lr_model, "Random Model": rndm_model}, X_test, y_test, X_train, y_train, idx_to_class)